In [48]:
#CheXper
from torch.utils.data import Dataset
from pathlib import Path
import pandas as pd
from PIL import Image
import os
import torch
from enum import Enum

In [2]:
DATASET_ROOT = os.path.abspath(os.path.join(".."))

# Dataset Generico

In [3]:
class NombreDataset(Dataset):
    
    def __init__(self, root,csv_path: str | Path, images_dir: str | Path, transform=None):
        self.root = root
        self.df = pd.read_csv(csv_path)
        self.images_dir = Path(images_dir)
        self.transform = transform
    
    def __len__(self) -> int:
        ...
    
    def __getitem__(self, idx: int) -> tuple:
        ...
    
    def _load_image(self, path: Path):
        ...
    
    def _extract_labels(self, row: pd.Series) -> dict:
        ...

# CheXpert

In [62]:
CHEXPERT_LABELS = [
"No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly",
"Lung Opacity", "Lung Lesion", "Edema", "Consolidation",
"Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion",
"Pleural Other", "Fracture"
]   

class ProjectionStrategy(Enum):
    # ¿Desde donde incide el rayo?
    PA_ONLY       = "PA" # El rayo entra por la espalda (Ideal - El paciente se encuentra parado)
    AP_ONLY       = "AP" # El rayo entra por el pecho (Notti - El paciente se encuentra acostado boca abajo)
    ALL           = "all"

class ViewStrategy(Enum):
    FRONTAL_ONLY  = "Frontal" # Siempre Frontal
    LATERAL_ONLY  = "Lateral"
    ALL           = "all"

#Si no se pasa Projection y View siempre se instancia Frontal y AP.

class CheXpertDataset(Dataset): 
    def __init__(self, root, csv_path: str | Path, images_dir: str | Path, 
                 transform=None, 
                 projection: ProjectionStrategy = ProjectionStrategy.AP_ONLY,
                 view: ViewStrategy = ViewStrategy.FRONTAL_ONLY):
        self.root = root
        self.df = pd.read_csv(csv_path).iloc[1:].reset_index(drop=True)
        self.df = self._filter_view(self.df, view)
        self.df = self._filter_projection(self.df, projection)
        self.images_dir = Path(images_dir)
        self.transform = transform

    def _filter_view(self, df: pd.DataFrame, strategy: ViewStrategy) -> pd.DataFrame:
        if strategy == ViewStrategy.ALL:
            return df
        return df[df["Frontal/Lateral"] == strategy.value].reset_index(drop=True)

    def _filter_projection(self, df: pd.DataFrame, strategy: ProjectionStrategy) -> pd.DataFrame:
        if strategy == ProjectionStrategy.ALL:
            return df
        return df[df["AP/PA"] == strategy.value].reset_index(drop=True)
    
    def __len__(self) -> int:
        return len(self.df)
    
    def __getitem__(self, idx: int) -> tuple:
        row = self.df.iloc[idx]
        try:
            image = self._load_image(row)
        except Exception:
            return self.__getitem__(idx + 1)
        labels = self._extract_labels(row)
        return image, labels
    
    def _load_image(self, row: pd.Series):
        img_path = row["Path"]
        image = Image.open(os.path.join(self.root,img_path)).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image
    
    def _extract_labels(self, row: pd.Series) -> dict:
        labels = {}
        for label in CHEXPERT_LABELS:
            value = row[label]
            if pd.isna(value):
                labels[label] = 0
            else:
                labels[label] = int(value)
        return labels
        

# ChestXray8 (Todas Freontales con AP/PA )

In [64]:
CX8_LABELS = [
    "Atelectasis", "Consolidation", "Infiltration", "Pneumothorax",
    "Edema", "Emphysema", "Fibrosis", "Effusion", "Pneumonia",
    "Pleural_Thickening", "Cardiomegaly", "Nodule", "Mass", "Hernia",
    "No Finding"
]

class ChestXray8Dataset(Dataset):

    def __init__(self, root, csv_path, split_ids=None, transform=None,
                 projection: ProjectionStrategy = ProjectionStrategy.AP_ONLY):
        self.root      = Path(root)
        self.transform = transform

        self.df = pd.read_csv(csv_path)
        self._path_map = self._build_path_map()

        if split_ids is not None:
            self.df = self.df[self.df["Image Index"].isin(set(split_ids))]

        self.df = self._filter_projection(self.df, projection)
        self.df = self.df.reset_index(drop=True)

    def _filter_projection(self, df: pd.DataFrame, strategy: ProjectionStrategy) -> pd.DataFrame:
        if strategy == ProjectionStrategy.ALL:
            return df
        return df[df["View Position"] == strategy.value].reset_index(drop=True)

    def _build_path_map(self) -> dict:
        path_map = {}
        cx8_root = self.root / "ChestXray8"
        for folder in sorted(os.listdir(cx8_root)):
            if folder.startswith("images_"):
                img_dir = cx8_root / folder / "images"
                if img_dir.is_dir():
                    for fname in os.listdir(img_dir):
                        path_map[fname] = img_dir / fname
        return path_map

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> tuple:
        row    = self.df.iloc[idx]
        image  = self._load_image(row)
        labels = self._extract_labels(row)
        return image, labels

    def _load_image(self, row: pd.Series):
        img_path = self._path_map.get(row["Image Index"])
        image    = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image

    def _extract_labels(self, row: pd.Series) -> dict:
        findings = set(row["Finding Labels"].split("|"))
        return {label: 1 if label in findings else 0 for label in CX8_LABELS}

# VinBigData (Solo para VAL)

In [66]:
VIN_LABELS = [
    "No finding", "Atelectasis", "Cardiomegaly", "Consolidation",
    "Pleural effusion", "Pleural thickening", "Pneumothorax",
    "Infiltration", "Nodule/Mass"
]

class VinBigDataset(Dataset):

    def __init__(self, root, csv_path, images_dir):
        self.root       = Path(root)
        self.images_dir = Path(images_dir)

        df = pd.read_csv(csv_path)
        votes = (
            df.groupby(["image_id", "class_name"])["rad_id"]
            .nunique().reset_index().rename(columns={"rad_id": "n_votes"})
        )
        n_rads = df.groupby("image_id")["rad_id"].nunique()
        votes["threshold"] = votes["image_id"].map(n_rads) * 0.5
        votes["positive"]  = (votes["n_votes"] >= votes["threshold"]).astype(float)

        all_images = df["image_id"].unique()
        self.df = (
            votes.pivot(index="image_id", columns="class_name", values="positive")
            .reindex(index=all_images, columns=VIN_LABELS)
            .fillna(0)
            .reset_index()
        )
        self.df.columns.name = None

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> tuple:
        row    = self.df.iloc[idx]
        image  = self._load_image(row)
        labels = self._extract_labels(row)
        return image, labels

    def _load_image(self, row: pd.Series):
        img_path = self.images_dir / f"{row['image_id']}.jpg"
        image    = Image.open(img_path).convert("RGB")
        return image

    def _extract_labels(self, row: pd.Series) -> dict:
        return {label: int(row[label]) for label in VIN_LABELS}

# UnifiedDataset

In [67]:
VIN_TO_CANONICAL = {
    "No finding":         "No_Finding",
    "Atelectasis":        "Atelectasis",
    "Cardiomegaly":       "Cardiomegaly",
    "Consolidation":      "Consolidation",
    "Pleural effusion":   "Effusion",
    "Pleural thickening": "Pleural_Thickening",
    "Pneumothorax":       "Pneumothorax",
    "Infiltration":       "Infiltration",
    "Nodule/Mass":        ["Nodule", "Mass"],  # uno-a-muchos
}

CANONICAL_LABELS = [
    "No_Finding",
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Effusion",
    "Emphysema",
    "Fibrosis",
    "Fracture",
    "Hernia",
    "Infiltration",
    "Lung_Lesion",
    "Lung_Opacity",
    "Nodule",
    "Mass",
    "Enlarged_Cardiomediastinum",
    "Pleural_Other",
    "Pleural_Thickening",
    "Pneumonia",
    "Pneumothorax",
]

CX8_TO_CANONICAL = {
    "Atelectasis":        "Atelectasis",
    "Consolidation":      "Consolidation",
    "Infiltration":       "Infiltration",
    "Pneumothorax":       "Pneumothorax",
    "Edema":              "Edema",
    "Emphysema":          "Emphysema",
    "Fibrosis":           "Fibrosis",
    "Effusion":           "Effusion",
    "Pneumonia":          "Pneumonia",
    "Pleural_Thickening": "Pleural_Thickening",
    "Cardiomegaly":       "Cardiomegaly",
    "Nodule":             "Nodule",
    "Mass":               "Mass",
    "Hernia":             "Hernia",
    "No Finding":         "No_Finding",
}

CHEXPERT_TO_CANONICAL = {
    "No Finding":                 "No_Finding",
    "Enlarged Cardiomediastinum": "Enlarged_Cardiomediastinum",
    "Cardiomegaly":               "Cardiomegaly",
    "Lung Opacity":               "Lung_Opacity",
    "Lung Lesion":                "Lung_Lesion",
    "Edema":                      "Edema",
    "Consolidation":              "Consolidation",
    "Pneumonia":                  "Pneumonia",
    "Atelectasis":                "Atelectasis",
    "Pneumothorax":               "Pneumothorax",
    "Pleural Effusion":           "Effusion",
    "Pleural Other":              "Pleural_Other",
    "Fracture":                   "Fracture",
}

In [68]:
class UnifiedDataset(Dataset):
    DATASET_EVALUATES = {
        "chexpert": set(CHEXPERT_TO_CANONICAL.values()),
        "cx8":      set(CX8_TO_CANONICAL.values()),
        "vin":      set(
            label
            for v in VIN_TO_CANONICAL.values()
            for label in (v if isinstance(v, list) else [v])
        ),
    }

    def __init__(self, datasets: dict, transform=None):
        self.datasets  = datasets
        self.transform = transform
        self._index_map = self._build_index_map()

    def _build_index_map(self) -> list:
        index_map = []
        for name, ds in self.datasets.items():
            print(name,ds)
            for i in range(len(ds)):
                index_map.append((name, i))
        return index_map

    def __len__(self) -> int:
        return len(self._index_map)

    def __getitem__(self, idx: int) -> tuple:
        name, local_idx = self._index_map[idx]
        image, raw_labels = self.datasets[name][local_idx]
        print('raw labels', raw_labels)

        mapping  = self._get_mapping(name)
        labels   = self._map_labels(raw_labels, mapping)

        labels_tensor = torch.tensor(
            [labels[l] for l in CANONICAL_LABELS], dtype=torch.float32
        )
        
        return image, labels_tensor

    def _get_mapping(self, name: str) -> dict:
        return {
            "chexpert": CHEXPERT_TO_CANONICAL,
            "cx8":      CX8_TO_CANONICAL,
            "vin":      VIN_TO_CANONICAL,
        }[name]

    def _map_labels(self, raw_labels: dict, mapping: dict) -> dict:
        canonical = {label: 0 for label in CANONICAL_LABELS} # Canonical con todos 0
        for pat, value in raw_labels.items():
            target = mapping.get(pat)
            if target is None:
                continue
            if isinstance(target, list):
                for t in target:
                    canonical[t] = value
            else:
                canonical[target] = value
        return canonical

# DataLoaders para train, validación y test

In [73]:
from torch.utils.data import random_split, DataLoader

chexpert = CheXpertDataset(
    root = DATASET_ROOT,
    csv_path   = os.path.join(DATASET_ROOT, "chexpert", "train.csv"),
    images_dir =  os.path.join(DATASET_ROOT,"chexpert","train"),
    projection=ProjectionStrategy.PA_ONLY,
    view=ViewStrategy.FRONTAL_ONLY
)
print(chexpert.__getitem__(1))

#Existencia de Frontal/Lateral y AP/PA
print(chexpert.df["Frontal/Lateral"].value_counts())
print(chexpert.df["AP/PA"].value_counts())

chest8 = ChestXray8Dataset(
    root = DATASET_ROOT,
    csv_path   = os.path.join(DATASET_ROOT, "ChestXray8", "Data_Entry_2017.csv"),
    projection=ProjectionStrategy.PA_ONLY
)
chest8.__getitem__(1) 

vinbig = VinBigDataset(
    root     = DATASET_ROOT,
    csv_path = os.path.join(DATASET_ROOT, "VinBigData", "train.csv"),
    images_dir =  os.path.join(DATASET_ROOT,"VinBigData", "train"),
)
vinbig.__getitem__(1)

unified = UnifiedDataset(datasets={
    "chexpert": chexpert,
    "cx8":      chest8,
    "vin":      vinbig,
})

unified.__getitem__(-1)
unified.__len__()

total = len(unified)
n_train = int(0.7 * total)
n_val   = int(0.15 * total)
n_test  = total - n_train - n_val

train_ds, val_ds, test_ds = random_split(unified, [n_train, n_val, n_test])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=4)

(<PIL.Image.Image image mode=RGB size=320x322 at 0x1E7F53D7BD0>, {'No Finding': 1, 'Enlarged Cardiomediastinum': 0, 'Cardiomegaly': 0, 'Lung Opacity': 0, 'Lung Lesion': 0, 'Edema': 0, 'Consolidation': 0, 'Pneumonia': 0, 'Atelectasis': 0, 'Pneumothorax': 0, 'Pleural Effusion': 0, 'Pleural Other': 0, 'Fracture': 0})
Frontal/Lateral
Frontal    29420
Name: count, dtype: int64
AP/PA
PA    29420
Name: count, dtype: int64
chexpert <__main__.CheXpertDataset object at 0x000001E7E588F9D0>
cx8 <__main__.ChestXray8Dataset object at 0x000001E7E588F750>
vin <__main__.VinBigDataset object at 0x000001E7EB0CB4D0>
raw labels {'No finding': 0, 'Atelectasis': 0, 'Cardiomegaly': 0, 'Consolidation': 0, 'Pleural effusion': 1, 'Pleural thickening': 1, 'Pneumothorax': 0, 'Infiltration': 0, 'Nodule/Mass': 0}
